# Investigation A — Weight decay and inherited adaptation geometry
**Inferential-adaptation-space-in-LLM** · unified research notebook

**Question.** Does the norm profile inherited from pretraining change the direction and usefulness of subsequent learning, rather than merely its speed? We separate the origin of the profile from its effects on a fixed recipient checkpoint. The core is Sections VI–VIII; the earlier sections establish experimental validity.

**Notation:** `[Algebra]` exact identities under stated assumptions; `[Theory]` literature or conditional mathematical results; `[Reported]` published observations; `[Proposed]` measurements and design decisions; `[Open]` unresolved empirical claims. No results are supplied in advance.

| Section | Question | Role |
|---|---|---|
| 0a | What are the checkpoints and protocol? | provenance |
| 0b | Does a native gap exist at our horizon? | relevance gate |
| I | What norm profiles are inherited? | intervention definition |
| II | What can decay exposure tell us? | optional historical context |
| III | Does the published LR grid cover compensation? | coverage audit |
| IV–V | Is the physical intervention valid and emulable? | identification |
| VI | Can global scaling reproduce the local update? | core geometry |
| VII | What granularity of scaling is needed? | core controls |
| VIII | Is non-global movement useful? | core utility |
| IX | Do effects persist across short trajectories? | robustness |
| X | Which hypotheses remain supported? | evidence ledger |

**Execution.** Place this file in `notebooks/` (or `notebooks/investigation_a/`), start the kernel from that directory or the repository root, and run top to bottom. The notebook downloads pinned public checkpoint revisions into `checkpoints/` and saves figures/tables to `results/investigation_a/`. Install a CUDA-enabled PyTorch build appropriate for the machine before running the dependency cell. Full-parameter FP32 experiments require a large GPU; no quantization, LoRA, or surrogate optimizer is used. Set the run configuration before collecting confirmatory results. The first execution is a local controlled study, not an exact reproduction of the paper's SFT pipeline. The public SimpleScaling full59K source is used with our own deterministic response formatting and splits, not asserted to match the authors' filtered 54,484 examples. A general-text WikiText proxy is not the original pretraining distribution.

References: [Han et al.](https://arxiv.org/html/2602.11137v2), [released weights](https://huggingface.co/collections/th135/pt), [training repository](https://github.com/th789/wd-plasticity), [Rotational Equilibrium](https://proceedings.mlr.press/v235/kosson24a.html), [AdamW timescale](https://arxiv.org/html/2405.13698v3), [Martens, natural gradient](https://jmlr.org/papers/v21/17-678.html), [effective LR in pretraining](https://arxiv.org/abs/2608.24814).

In [ ]:
# Run once in the intended environment, then restart the kernel if needed.
%pip install -q "transformers==4.55.3" "huggingface_hub>=0.34,<1" "datasets>=3.6,<5" safetensors accelerate scipy pandas matplotlib seaborn tqdm tabulate

In [ ]:
from pathlib import Path
from collections import defaultdict
from contextlib import contextmanager
import gc, json, math, re, hashlib, platform, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import nnls
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from huggingface_hub import HfApi, snapshot_download
from safetensors import safe_open
from datasets import load_dataset
from IPython.display import display, Markdown

# Relative paths, including when the notebook is nested in notebooks/investigation_a.
ROOT = Path('.') if Path('.git').exists() else Path('..') if Path('../.git').exists() else Path('../..')
OUT = ROOT / 'results/investigation_a'
CKPT = ROOT / 'checkpoints'
OUT.mkdir(parents=True, exist_ok=True)
CKPT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32
if DEVICE.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)

MODELS = {0.0:'sbordt/OLMo-2-1B-1x-WD0', 0.1:'sbordt/OLMo-2-1B-1x',
          0.3:'sbordt/OLMo-2-1B-1x-WD03', 0.6:'sbordt/OLMo-2-1B-1x-WD06',
          1.0:'sbordt/OLMo-2-1B-1x-WD1'}
PRIMARY = (0.1, 0.6, 1.0)
PAIRS = [(0.1,1.0),(0.6,1.0),(1.0,0.1),(1.0,0.6)] # recipient, donor
ELIGIBLE = ('q_proj','k_proj','v_proj','o_proj','up_proj','down_proj')
FAMILIES = list(ELIGIBLE) + ['gate_proj','q_norm','k_norm','post_attention_layernorm','post_feedforward_layernorm','embed_tokens','norm','lm_head']
TASKS = ('math','race')
SEEDS = (11, 23, 37)
DATA_SEED = 2026
MAX_LEN = 512
MICRO_BATCH = 4
TRAIN_BATCH = 16  # custom local protocol; same examples and token masks across arms
DIAG_BATCH = 8
DIAG_REPEATS = 8
EVAL_EXAMPLES = 64
CONTEXT_EXAMPLES = 8
GEOMETRY_TOKENS = 256 # per context partition
ANCHOR_STEPS = 500
ANCHOR_POINTS = (0,1,10,25,50,100,250,500)
LR_GRID = (1e-5,3e-5,6e-5,1e-4)
BASE_LR = 1e-5
BETAS = (0.9,0.95)
EPS = 1e-8
FT_WD = 0.0
# Constant LR in this local protocol; this is not a compressed paper schedule.
FD_H = 1.0
FD_SCALES = (0.25,0.5,1.,2.)
N_SHUFFLES = 200
STEP_SCALES = (0.25,0.5,1.,2.)
# Provisional, declare before evaluation; gap-relative utility margin calibrated below.
EQUIV_RESIDUAL = .05
ANCHOR_MIN_GAP = .01 # nats/token, minimum useful native gap, not a Fisher threshold
GAP_FRACTION = .10
SYMMETRY_KL_LIMIT = 1e-7 # provisional absolute cap, also compare against intended step KL
SYMMETRY_GRAD_LIMIT = .02
SYMMETRY_STEP_FRACTION = .05
IDENTITY_PARAMETER_TOL = 1e-5 # provisional relative transported-parameter error
IDENTITY_STEPS = 10
TRAJECTORY_POINTS = (0,1,10,50,100,250,500)
RUN_SWEEP_REEVAL = False # conditional: requires a verified mapping, see III
RUN_HISTORICAL = True
RUN_FP32_SYMMETRY = True
RUN_BF16_SENSITIVITY = True
RUN_TRAJECTORIES = True
RUN_BENCHMARK = True # GSM8K numeric exact-match diagnostic; not paper benchmark reproduction
BENCHMARK_N = 64

settings = {k:v for k,v in globals().copy().items() if k in ['PRIMARY','PAIRS','TASKS','SEEDS','DATA_SEED','MAX_LEN','MICRO_BATCH','TRAIN_BATCH','DIAG_BATCH','DIAG_REPEATS','EVAL_EXAMPLES','CONTEXT_EXAMPLES','GEOMETRY_TOKENS','ANCHOR_STEPS','LR_GRID','BASE_LR','BETAS','EPS','FT_WD','FD_H','FD_SCALES','N_SHUFFLES','STEP_SCALES','EQUIV_RESIDUAL','ANCHOR_MIN_GAP','GAP_FRACTION','SYMMETRY_KL_LIMIT','SYMMETRY_GRAD_LIMIT','SYMMETRY_STEP_FRACTION','IDENTITY_PARAMETER_TOL','IDENTITY_STEPS','TRAJECTORY_POINTS','RUN_SWEEP_REEVAL','RUN_HISTORICAL','RUN_FP32_SYMMETRY','RUN_BF16_SENSITIVITY','RUN_TRAJECTORIES','RUN_BENCHMARK','BENCHMARK_N']}
settings.update(torch=torch.__version__, python=platform.python_version(), dtype=str(DTYPE), device=str(DEVICE))
(OUT/'experiment_config.json').write_text(json.dumps(settings,indent=2))

def save_table(df, name):
    df.to_csv(OUT/f'{name}.csv', index=False)
    return df

def finish(fig, name):
    fig.tight_layout()
    fig.savefig(OUT/f'{name}.png', dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

def cleanup():
    gc.collect()
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()

def seed_all(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(seed)

## 0a. Checkpoint provenance and experimental setup
[Reported] The study uses 20 TPP OLMo-2 weights across WD values. The WD=1 public `config.json` gives 16 layers, hidden size 2048, intermediate size **8192**, 16 query and KV heads, vocabulary **100352**, RMSNorm epsilon **1e-6**, no attention bias or dropout, untied embedding/head. These are checkpoint configuration values, not the earlier paper-table estimates. Read metadata for the exact pinned revisions; unknown initialization/data order or decay policies remain unknown. No dimension assertions or generalized architecture adapters.

[Proposed] A recipient–donor pair changes only the recipient's eligible scalar coordinates or optimizer. The independently trained donor is not assumed functionally equivalent. Embedding cosine, if examined, is a coordinate-level descriptive check only.

In [ ]:
api = HfApi()
paths, manifest = {}, []
for wd, repo in tqdm(MODELS.items(), desc='Pinned checkpoints'):
    revision = api.model_info(repo).sha
    path = Path(snapshot_download(repo_id=repo, revision=revision,
                                  local_dir=CKPT/f'wd{wd:g}',
                                  allow_patterns=['*.safetensors','*.json','*.model','*.txt','*.jinja']))
    paths[wd] = path
    cfg = AutoConfig.from_pretrained(path).to_dict()
    manifest.append(dict(wd=wd, repo=repo, revision=revision, config=cfg,
                         initialization_seed='unknown', data_order='unknown',
                         decay_policy='input embedding excluded (reported); others unverified'))
(OUT/'checkpoint_manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
fields = ['model_type','num_hidden_layers','hidden_size','intermediate_size','num_attention_heads','num_key_value_heads','vocab_size','rms_norm_eps','tie_word_embeddings']
display(pd.DataFrame([{'wd':m['wd'],**{k:m['config'][k] for k in fields}} for m in manifest]))
tokenizer = AutoTokenizer.from_pretrained(paths[0.1])
PAD, EOS = tokenizer.pad_token_id or manifest[1]["config"]["pad_token_id"], tokenizer.eos_token_id or manifest[1]["config"]["eos_token_id"]

def load_model(wd, dtype=DTYPE):
    model = AutoModelForCausalLM.from_pretrained(paths[wd], torch_dtype=dtype,
                                                attn_implementation='eager').to(DEVICE)
    model.config.use_cache = False
    return model.eval() # dropout is zero; autograd remains enabled

def family(name):
    return name.split('.')[-2]

def layer(name):
    match = re.search(r'model\.layers\.(\d+)\.',name)
    return int(match.group(1)) if match else -1

def candidate(name):
    return family(name) in ELIGIBLE

def block_name(name):
    # Fine-grained eligible matrices, remaining weights grouped by actual family.
    return name if candidate(name) else 'other.'+family(name)

def groups_of(model):
    groups = defaultdict(list)
    for name,p in model.named_parameters(): groups[block_name(name)].append(name)
    return dict(groups)

### Data partitions and local training protocol
[Proposed] Math uses the released `simplescaling/data_ablation_full59K` question, thinking trajectories and attempt fields. Use a deterministic streaming shuffle, deduplicate exact normalized questions, discard examples with no supervised response, and divide by question into training, calibration and final evaluation partitions. RACE uses official train/validation/test (disjoint article splits); train answers as letter tokens. WikiText-103 validation/test are retention calibration/evaluation proxies. The SFT response begins after the question; truncate long reasoning to the sequence budget (not paper reproduction). Retention masks all nonpadding target tokens. Separate diagnostic gradient pools are reserved from the training source, disjoint from training and from each other. All comparisons use identical example ordering within a task/seed; count supervised tokens and processed tokens. The geometry uses disjoint groups of held-out examples across diagnostic repeats and fixed sampled target contexts with equal weight. Exact question deduplication is not semantic deduplication; benchmark contamination cannot be ruled out. No final evaluation data select learning rates or group coefficients.

In [ ]:
def encode(prompt, response=None):
    if response is None:
        ids = tokenizer.encode(prompt, add_special_tokens=False)[:MAX_LEN]
        labels = ids.copy()
    else:
        prefix = tokenizer.encode(prompt, add_special_tokens=False)[:MAX_LEN//2]
        suffix = tokenizer.encode(response, add_special_tokens=False)
        room = MAX_LEN-len(prefix)
        # Do not append EOS to a truncated response.
        suffix = (suffix+[EOS])[:room]
        ids = prefix+suffix
        labels = [-100]*len(prefix)+suffix
    return {'ids':ids, 'labels':labels}

def math_records(n):
    ds = load_dataset('simplescaling/data_ablation_full59K', split='train', streaming=True).shuffle(seed=DATA_SEED,buffer_size=10000)
    records, seen = [], set()
    for row in ds:
        question = row['question'].strip()
        attempt = (row['attempt'] or '').strip()
        trace = '\n'.join(str(t) for t in (row['thinking_trajectories'] or []) if t)
        key = ' '.join(question.lower().split())
        if key in seen or not question or not attempt: continue
        seen.add(key)
        text = (trace+'\n'+attempt).strip()
        records.append((hashlib.sha256(key.encode()).hexdigest(),encode('Question: '+question+'\nAnswer: ',text)))
        if len(records)==n: break
    return records

def race_records(split,n):
    ds = load_dataset('ehovy/race','all',split=split).shuffle(seed=DATA_SEED)
    result = []
    for row in ds.select(range(min(n,len(ds)))):
        choices = '\n'.join(f'{chr(65+i)}. {s}' for i,s in enumerate(row['options']))
        prompt = row['article']+'\nQuestion: '+row['question']+'\n'+choices+'\nAnswer: '
        # Keep the question and choices when the article is long.
        tail = tokenizer.encode('\nQuestion: '+row['question']+'\n'+choices+'\nAnswer: ',add_special_tokens=False)
        article = tokenizer.encode(row['article'],add_special_tokens=False)[:max(0,MAX_LEN//2-len(tail))]
        ids = (article+tail)[-MAX_LEN//2:]
        answer = tokenizer.encode(row['answer'],add_special_tokens=False)+[EOS]
        result.append((row['example_id'],{'ids':ids+answer,'labels':[-100]*len(ids)+answer}))
    return result

def retention_records(split,n):
    ds = load_dataset('Salesforce/wikitext','wikitext-103-raw-v1',split=split)
    result=[]
    for i,row in enumerate(ds):
        if len(row['text'].split())<30: continue
        result.append((str(i),encode(row['text'])))
        if len(result)==n: break
    return result

n_train = max(4096,TRAIN_BATCH*ANCHOR_STEPS)
n_diag = DIAG_BATCH*DIAG_REPEATS
math_data = math_records(n_train+2*EVAL_EXAMPLES+2*n_diag)
DATA = {'math':{'train':math_data[:n_train], 'cal':math_data[n_train:n_train+EVAL_EXAMPLES], 'eval':math_data[n_train+EVAL_EXAMPLES:n_train+2*EVAL_EXAMPLES], 'grad_cal':math_data[n_train+2*EVAL_EXAMPLES:n_train+2*EVAL_EXAMPLES+n_diag], 'grad_eval':math_data[-n_diag:]},
        'race':{'train':race_records('train',n_train+2*n_diag),'cal':race_records('validation',EVAL_EXAMPLES),'eval':race_records('test',EVAL_EXAMPLES)},
        'retention':{'cal':retention_records('validation',EVAL_EXAMPLES),'eval':retention_records('test',EVAL_EXAMPLES)}}
race_train = DATA['race']['train']
DATA['race']['train'] = race_train[:n_train]
DATA['race']['grad_cal'] = race_train[n_train:n_train+n_diag]
DATA['race']['grad_eval'] = race_train[n_train+n_diag:]
(OUT/'data_manifest.json').write_text(json.dumps({t:{s:[i for i,_ in rows] for s,rows in parts.items()} for t,parts in DATA.items()},indent=2))

def batches(rows, size, seed=None):
    order = np.arange(len(rows)) if seed is None else np.random.default_rng(seed).permutation(len(rows))
    for k in range(0,len(order),size):
        yield [rows[i][1] for i in order[k:k+size]]

def diagnostic_batches(task,split):
    return batches(DATA[task]['grad_'+split], DIAG_BATCH)

def training_batches(task, size, seed):
    epoch=0
    while True:
        yield from batches(DATA[task]['train'],size,seed+epoch)
        epoch+=1

def collate(rows):
    length = max(len(r['ids']) for r in rows)
    ids = torch.full((len(rows),length),PAD,dtype=torch.long)
    labels = torch.full_like(ids,-100)
    mask = torch.zeros_like(ids)
    for i,r in enumerate(rows):
        n=len(r['ids']); ids[i,:n]=torch.tensor(r['ids']); labels[i,:n]=torch.tensor(r['labels']); mask[i,:n]=1
    return ids.to(DEVICE),mask.to(DEVICE),labels.to(DEVICE)

def selected_logits(model,batch,positions=None):
    ids,mask,labels = batch
    h = model.model(input_ids=ids,attention_mask=mask,use_cache=False).last_hidden_state[:,:-1]
    valid = labels[:,1:]!=-100
    h,y = h[valid],labels[:,1:][valid]
    if positions is not None: h,y=h[positions],y[positions]
    return model.lm_head(h).float(),y

def backward_batch(model, rows):
    model.zero_grad(set_to_none=True)
    count = sum(sum(y!=-100 for y in r['labels'][1:]) for r in rows)
    total=0.
    for k in range(0,len(rows),MICRO_BATCH):
        logits,y = selected_logits(model,collate(rows[k:k+MICRO_BATCH]))
        loss=F.cross_entropy(logits,y,reduction='sum')
        (loss/count).backward()
        total+=loss.item()
    return total/count, count, sum(len(r['ids']) for r in rows)

@torch.no_grad()
def eval_loss(model,task,split):
    total=count=correct=0
    for rows in batches(DATA[task][split],MICRO_BATCH):
        z,y=selected_logits(model,collate(rows))
        total+=F.cross_entropy(z,y,reduction='sum').item(); count+=y.numel()
        correct+=(z.argmax(-1)==y).sum().item()
    return total/count,correct/count

def optimizer(model, lr, profile=None, mode='lr_only', wd=FT_WD, eps=EPS):
    groups=[]
    for name,p in model.named_parameters():
        a=1. if profile is None else profile[name]
        # a=1/c, exact emulation sets epsilon and decay to c times their base values.
        e=eps/a if mode=='exact' else eps
        decay=wd/a if mode=='exact' else wd
        groups.append({'params':[p],'lr':lr*a,'eps':e,'weight_decay':decay})
    return torch.optim.AdamW(groups, betas=BETAS, foreach=False)

def bootstrap_mean(x,seed=123,n=4000):
    x=np.asarray(x,dtype=float); x=x[np.isfinite(x)]
    if not len(x): return np.nan,np.nan,np.nan
    samples=np.random.default_rng(seed).choice(x,(n,len(x)),replace=True).mean(1)
    return x.mean(),*np.quantile(samples,[.025,.975])

## 0b. Native adaptation anchor — relevance gate
[Open] Is there a positive native short-horizon gap for H versus L and M? Define $P_s(t)=L_s(0)-L_s(t)$, $
\Delta(t)=P_H(t)-P_s(t)$. Also track absolute loss, retention loss and a numeric GSM8K diagnostic (not GSM8KPlatinum/MATH reproduction). LR calibration uses a separate seed and calibration split, never the final evaluation split. Shared LR and independently calibrated LR are both reported. An absent or inconclusive anchor limits causal relevance claims; it does not invalidate the local mathematical experiment. The primary anchor is 500 steps under the constant-LR protocol; all native/intervention curves share the token budget and data order. The calibration gap determines a separate practical utility margin; never use it directly as a Fisher tolerance.

In [ ]:
if RUN_BENCHMARK:
    gsm = load_dataset('openai/gsm8k','main',split='test').shuffle(seed=DATA_SEED).select(range(BENCHMARK_N))

def numeric_answer(text):
    matches=re.findall(r'[-+]?\d+(?:\.\d+)?',text.replace(',',''))
    return float(matches[-1]) if matches else None

@torch.no_grad()
def gsm_accuracy(model):
    correct=0
    for row in gsm:
        prompt='Question: '+row['question']+'\nAnswer: '
        ids=tokenizer(prompt,return_tensors='pt',add_special_tokens=False).input_ids.to(DEVICE)
        gen=model.generate(ids,attention_mask=torch.ones_like(ids),max_new_tokens=256,do_sample=False,pad_token_id=PAD,eos_token_id=EOS,use_cache=True)
        answer=tokenizer.decode(gen[0,ids.shape[1]:],skip_special_tokens=True)
        target=numeric_answer(row['answer'].split('####')[-1]); predicted=numeric_answer(answer.split('####')[-1])
        correct+=predicted is not None and predicted==target
    return correct/len(gsm)

def native_run(wd,task,lr,seed,split,points,benchmark=False):
    seed_all(seed); model=load_model(wd); opt=optimizer(model,lr)
    stream=training_batches(task,TRAIN_BATCH,seed)
    records=[]; seen=supervised=0
    initial=eval_loss(model,task,split)[0]
    ret0=eval_loss(model,'retention',split)[0]
    for step in range(max(points)+1):
        if step in points:
            loss,acc=eval_loss(model,task,split); ret=eval_loss(model,'retention',split)[0]
            gen=gsm_accuracy(model) if benchmark and task=='math' and step in (0,max(points)) else np.nan
            records.append(dict(wd=wd,task=task,lr=lr,seed=seed,split=split,step=step,
                                tokens=seen,supervised_tokens=supervised,loss=loss,improvement=initial-loss,
                                token_accuracy=acc,retention_change=ret-ret0,gsm8k_numeric_accuracy=gen))
        if step==max(points): break
        _,n,t=backward_batch(model,next(stream)); opt.step(); supervised+=n; seen+=t
    del opt,model; cleanup()
    return records

cal_records=[]
for task in TASKS:
    for wd in PRIMARY:
        for lr in LR_GRID:
            cal_records.extend(native_run(wd,task,lr,DATA_SEED,'cal',(0,100,ANCHOR_STEPS)))
cal=save_table(pd.DataFrame(cal_records),'native_lr_calibration')
end=cal[cal.step==ANCHOR_STEPS]
best_lr=end.loc[end.groupby(['task','wd']).loss.idxmin()].set_index(['task','wd']).lr.to_dict()
display(end.pivot_table(index=['task','wd'],columns='lr',values='loss'))
anchor_records=[]
for task in TASKS:
    for wd in PRIMARY:
        for seed in SEEDS:
            for lr in sorted({BASE_LR,best_lr[task,wd]}):
                rows=native_run(wd,task,lr,seed,'eval',ANCHOR_POINTS,RUN_BENCHMARK)
                for r in rows: r['shared_lr']=lr==BASE_LR; r['tuned_lr']=lr==best_lr[task,wd]
                anchor_records.extend(rows)
anchor=save_table(pd.DataFrame(anchor_records),'native_adaptation_anchor')

In [ ]:
# Margins from calibration only, then evaluate anchor on independent seeds/split.
margin={}; anchor_decisions=[]
for task in TASKS:
    c=end[end.task==task].set_index(['wd','lr']).improvement
    for lo in (.1,.6):
        gap=c.loc[(1.,best_lr[task,1.])]-c.loc[(lo,best_lr[task,lo])]
        margin[task,lo]=max(ANCHOR_MIN_GAP,GAP_FRACTION*max(0,gap))
        for arm in ['shared_lr','tuned_lr']:
            subset=anchor[(anchor.task==task)&anchor[arm]&(anchor.step==ANCHOR_STEPS)]
            piv=subset.pivot(index='seed',columns='wd',values='improvement')
            estimate,low,high=bootstrap_mean(piv[1.]-piv[lo])
            decision='positive gap' if low>ANCHOR_MIN_GAP else 'equivalent' if low>=-ANCHOR_MIN_GAP and high<=ANCHOR_MIN_GAP else 'negative gap' if high< -ANCHOR_MIN_GAP else 'inconclusive'
            anchor_decisions.append(dict(task=task,recipient=lo,arm=arm,estimate=estimate,ci_low=low,ci_high=high,utility_margin=margin[task,lo],decision=decision))
anchor_gate=save_table(pd.DataFrame(anchor_decisions),'anchor_decisions')
display(anchor_gate)
for task in TASKS:
    fig,ax=plt.subplots(1,3,figsize=(15,4))
    subset=anchor[(anchor.task==task)&anchor.tuned_lr]
    for wd,grp in subset.groupby('wd'):
        stats=grp.groupby('step')[['loss','improvement','retention_change']].mean()
        for axis,col in zip(ax,stats.columns):
            axis.plot(stats.index,stats[col],marker='o',label=f'WD {wd:g}'); axis.set(xlabel='Optimizer step',ylabel=col)
            for _,run in grp.groupby('seed'): axis.plot(run.step,run[col],alpha=.18)
    ax[0].legend(); finish(fig,f'anchor_{task}')

## I. Static parameter geometry — define the intervention
[Algebra] $r_i=\|W_i\|_F$, RMS $=r_i/\sqrt{N_i}$, squared-norm share $=r_i^2/\sum_jr_j^2$. For recipient $s$, donor $h$, $c_i=r_i^h/r_i^s$, $a_i=1/c_i$ on the six candidate projection families; elsewhere $a_i=1$. These are *intervention coefficients*, not measured differences in learning rate between independent models. Report heterogeneity descriptively, not as a significance test across layers. Retain effective output head $W_{\rm head}\operatorname{diag}(\gamma)$, which can be measured by columnwise multiplication without constructing a diagonal matrix.

In [ ]:
def tensor_index(path):
    return json.loads((path/'model.safetensors.index.json').read_text())['weight_map']

def read_tensor(path,index,name):
    with safe_open(path/index[name],framework='pt',device='cpu') as f:
        return f.get_tensor(name)

norm_rows=[]; head_rows=[]
for wd,path in tqdm(paths.items(),desc='Static geometry'):
    idx=tensor_index(path)
    for shard in sorted(set(idx.values())):
        with safe_open(path/shard,framework='pt',device='cpu') as f:
            for name in f.keys():
                if not name.endswith('.weight'): continue
                w=f.get_tensor(name)
                r=torch.linalg.vector_norm(w.float()).item()
                norm_rows.append(dict(wd=wd,parameter=name,layer=layer(name),family=family(name),
                                      shape=str(tuple(w.shape)),numel=w.numel(),dtype=str(w.dtype),norm=r,
                                      rms=r/math.sqrt(w.numel()),candidate=candidate(name),
                                      directly_decayed='no (reported)' if family(name)=='embed_tokens' else 'unknown'))
    gamma=read_tensor(path,idx,'model.norm.weight').float()
    head=read_tensor(path,idx,'lm_head.weight')
    col_energy=torch.zeros(gamma.numel(),dtype=torch.float64)
    for chunk in head.split(2048): col_energy+=chunk.float().square().sum(0,dtype=torch.float64)
    head_rows.append(dict(wd=wd,head_norm=col_energy.sum().sqrt().item(),gain_rms=gamma.square().mean().sqrt().item(),
                          effective_norm=(col_energy*gamma.double().square()).sum().sqrt().item()))
    del head,gamma
norms=pd.DataFrame(norm_rows)
norms['norm_sq']=norms.norm**2
norms['share']=norms.norm_sq/norms.groupby('wd').norm_sq.transform('sum')
save_table(norms,'parameter_geometry')
save_table(norms[norms.wd==.1].drop(columns=['wd','norm','rms','norm_sq','share']),'parameter_inventory')
head_geometry=save_table(pd.DataFrame(head_rows),'effective_head_geometry')
lookup=norms.pivot(index='parameter',columns='wd',values='norm')
profile_rows=[]; profiles={}
for recipient,donor in PAIRS:
    ratio=lookup[recipient]/lookup[donor]
    profiles[recipient,donor]={name:float(v) if candidate(name) else 1. for name,v in ratio.items()}
    for name,v in ratio.items():
        profile_rows.append(dict(recipient=recipient,donor=donor,parameter=name,layer=layer(name),family=family(name),
                                 candidate=candidate(name),norm_ratio=v,c=1/v if candidate(name) else 1.,
                                 adam=profiles[recipient,donor][name],sgd=profiles[recipient,donor][name]**2))
profile_df=save_table(pd.DataFrame(profile_rows),'norm_emulation_profiles')
summary=profile_df[profile_df.candidate].groupby(['recipient','donor','family']).adam.agg(['min','median','max'])
display(summary)

In [ ]:
fig,axes=plt.subplots(2,4,figsize=(15,7),sharex=True)
for ax,fam in zip(axes.flat,list(ELIGIBLE)+['gate_proj','post_attention_layernorm']):
    for wd,g in norms[norms.family==fam].groupby('wd'):
        g=g.sort_values('layer'); ax.plot(g.layer,g.rms,marker='.',label=f'{wd:g}')
    ax.set(title=fam,xlabel='Layer',ylabel='Weight RMS')
axes[0,0].legend(title='WD'); finish(fig,'absolute_rms_profiles')
fig,axes=plt.subplots(2,2,figsize=(16,8))
limit=np.abs(np.log2(profile_df[profile_df.candidate].norm_ratio)).max()
for ax,(pair,g) in zip(axes.flat,profile_df[profile_df.candidate].groupby(['recipient','donor'])):
    mat=np.log2(g.pivot(index='family',columns='layer',values='norm_ratio').reindex(ELIGIBLE))
    sns.heatmap(mat,ax=ax,cmap='vlag',center=0,vmin=-limit,vmax=limit,annot=False,cbar_kws={'label':'log2 norm ratio'})
    ax.set_title(f'{pair[0]:g} receives {pair[1]:g}')
finish(fig,'norm_ratio_profiles')
ref=norms[norms.wd==.1].set_index('parameter').norm
norms['relative_norm']=norms.norm/norms.parameter.map(ref)
fig,ax=plt.subplots(figsize=(10,5))
for fam,g in norms.groupby('family'):
    s=g.groupby('wd').relative_norm.agg(['median','min','max']); x=s.index.to_numpy(float)
    ax.plot(x,s['median'],marker='o',label=fam); ax.fill_between(x,s['min'],s['max'],alpha=.07)
ax.set(xlabel='Pretraining WD',ylabel='Norm / WD 0.1 norm',title='Dose response; range across layers (not seed CI)'); ax.legend(ncol=2,fontsize=8); finish(fig,'norm_dose_response')
fig,ax=plt.subplots(figsize=(8,4))
head_geometry.set_index('wd').div(head_geometry.set_index('wd').loc[.1]).plot(marker='o',ax=ax)
ax.set(ylabel='Relative to WD 0.1',xlabel='Pretraining WD'); finish(fig,'effective_head')
fig,ax=plt.subplots(figsize=(10,4))
norms.pivot_table(index='wd',columns='family',values='share',aggfunc='sum').plot.bar(stacked=True,ax=ax)
ax.set(ylabel='Share of total squared norm'); ax.legend(bbox_to_anchor=(1,1),fontsize=8); finish(fig,'norm_share')

## II. Historical norm dynamics (optional context, not a primary hypothesis)
[Algebra] AdamW: $W_{t+1}=c_t W_t-\eta_t A_t$, $c_t=1-\eta_t\lambda$. Then $q_{t+1}=c_t^2q_t+\eta_t^2\|A_t\|^2-2\eta_t c_t\langle W_t,A_t\rangle$. Adam's preconditioning and historical moments can create radial components even for scale-invariant weights. Under a simplified stationary autocorrelation model with $\mathbb E[A_t^\top A_{t-j}]=B\beta_1^j$, the effective update energy becomes $B(1+c\beta_1)/(1-c\beta_1)$. The original zero-cross-term model is withdrawn. Endpoint norms cannot identify those terms, or prove equilibrium. The following reconstructs the reported schedule approximately and calculates *initialization attenuation*, not a fraction of final norm. A long relaxation time is not proof of nonequilibrium.

In [ ]:
if RUN_HISTORICAL:
    steps=round(30e9/(512*4096)); warm=round(.1*steps)
    schedule=np.r_[np.linspace(0,4e-4,warm,endpoint=False),4e-5+.5*(4e-4-4e-5)*(1+np.cos(np.linspace(0,np.pi,steps-warm)))]
    hist=[]
    for wd in MODELS:
        hist.append(dict(wd=wd,steps=steps,integrated_lr=schedule.sum(),decay_exposure=wd*schedule.sum(),
                         initialization_sq_attenuation=np.exp(2*np.log1p(-wd*schedule).sum()),
                         final_relaxation_steps=-1/(2*np.log1p(-wd*schedule[-1])) if wd else np.inf))
    historical=save_table(pd.DataFrame(hist),'decay_exposure'); display(historical)
    fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].plot(schedule); ax[0].set(xlabel='Step',ylabel='Reconstructed LR')
    for wd in MODELS:
        ax[1].plot(np.exp(2*np.cumsum(np.log1p(-wd*schedule))),label=f'{wd:g}')
    ax[1].set(xlabel='Step',ylabel='Squared initialization attenuation'); ax[1].legend(title='WD'); finish(fig,'decay_exposure')

## III. Published LR evidence — coverage audit
[Reported] Appendix E.2 uses fine-tuning LRs $10^{-5},3\times10^{-5},6\times10^{-5}$, with a finite grid of other hyperparameters. [Proposed] Compute the predicted common multiplier as the geometric mean over eligible blocks; it does not imply the whole model scales uniformly. The paper's aggregate maxima do not establish a continuous optimum. Compare with the independent native calibration curves from 0b. An optional input CSV `results/investigation_a/published_sweep_manifest.csv` with columns `repo,wd,lr,batch_size,ft_wd,task` enables re-evaluation of a manually verified mapping of the authors' [released sweep](https://huggingface.co/collections/th135/vary-wd-lr-bs-during-ft). Do not infer batch size or dataset from ambiguous repository names. This conditional experiment is disabled until the mapping is verified.

In [ ]:
coverage=[]
for (recipient,donor),profile in profiles.items():
    a=np.array([v for k,v in profile.items() if candidate(k)])
    global_a=np.exp(np.log(a).mean()); gsd=np.exp(np.log(a).std())
    for eta in (1e-5,3e-5,6e-5):
        required=global_a*eta
        coverage.append(dict(recipient=recipient,donor=donor,geometric_mean=global_a,gsd=gsd,donor_lr=eta,
                             predicted_recipient_lr=required,inside_grid=1e-5<=required<=6e-5))
coverage=save_table(pd.DataFrame(coverage),'published_lr_coverage'); display(coverage)
for task in TASKS:
    fig,ax=plt.subplots(figsize=(8,4))
    for wd,g in end[end.task==task].groupby('wd'):
        ax.plot(g.lr,g.loss,marker='o',label=f'WD {wd:g}')
    ax.axvspan(1e-5,6e-5,alpha=.09,label='Published LR interval'); ax.set(xscale='log',xlabel='LR',ylabel='Calibration loss at 500 steps'); ax.legend(); finish(fig,f'lr_calibration_{task}')
if RUN_SWEEP_REEVAL:
    sweep=pd.read_csv(OUT/'published_sweep_manifest.csv'); sweep_rows=[]
    for r in sweep.itertuples(index=False):
        revision=api.model_info(r.repo).sha
        model=AutoModelForCausalLM.from_pretrained(r.repo,revision=revision,torch_dtype=DTYPE,attn_implementation='eager').to(DEVICE).eval()
        loss,acc=eval_loss(model,r.task,'cal')
        sweep_rows.append(dict(repo=r.repo,revision=revision,wd=r.wd,lr=r.lr,batch_size=r.batch_size,ft_wd=r.ft_wd,task=r.task,loss=loss,token_accuracy=acc))
        del model; cleanup()
    sweep_results=save_table(pd.DataFrame(sweep_rows),'published_sweep_reevaluation')
    display(sweep_results)

## IV. Function-preserving intervention validation
[Theory] Positive scalar scaling of Q/K or V/O/up/down is approximately invariant through the OLMo-2 normalization placement, not exact with finite RMSNorm epsilon. In contrast reciprocal up/down scaling cancels algebraically in SwiGLU. Validate the function and gradients independently; a small KL does not imply gradient covariance. Positive control: exact reciprocal up/down pair; negative control: gate projection. Compare repeated originals, identity, each family, and full profile in FP32 and BF16. Use a provisional absolute KL cap *and* an intended-step-relative criterion calibrated on separate contexts; a rejected candidate may still be used in an LR intervention but not called a validated gauge transplant.

In [ ]:
def make_context(task,split,seed=DATA_SEED):
    records=DATA[task][split]
    order=np.random.default_rng(DATA_SEED).permutation(len(records))
    offset=(seed-DATA_SEED)*CONTEXT_EXAMPLES
    rows=[records[i][1] for i in order[offset:offset+CONTEXT_EXAMPLES]]
    batch=collate(rows)
    n=int((batch[2][:,1:]!=-100).sum())
    positions=torch.from_numpy(np.sort(np.random.default_rng(seed).choice(n,min(n,GEOMETRY_TOKENS),replace=False))).to(DEVICE)
    return batch,positions

@torch.no_grad()
def context_logits(model,ctx):
    return selected_logits(model,*ctx)[0].detach()

def context_targets(ctx):
    batch,positions=ctx
    labels=batch[2][:,1:]
    return labels[labels!=-100][positions]

def centered(z): return z-z.mean(-1,keepdim=True)

def kl(z0,z1):
    # Float64 avoids cancellation when KL is extremely small.
    p=z0.double().log_softmax(-1); q=z1.double().log_softmax(-1)
    return (p.exp()*(p-q)).sum(-1).mean().item()

def output_metrics(z0,z1):
    diff=z1-z0
    return dict(kl=kl(z0,z1),max_logit_error=diff.abs().max().item(),
                centered_relative_error=(centered(diff).norm()/centered(z0).norm()).item())

@contextmanager
def scaled_weights(model,scales):
    named=dict(model.named_parameters()); saved={}
    with torch.no_grad():
        for name,c in scales.items():
            if c==1: continue
            p=named[name]; saved[name]=p.detach().clone(); p.mul_(c)
    try: yield
    finally:
        with torch.no_grad():
            for name,value in saved.items(): named[name].copy_(value)

@contextmanager
def perturbed(model,updates,scale=1.):
    named=dict(model.named_parameters()); saved={}
    with torch.no_grad():
        for name,v in updates.items():
            p=named[name]; saved[name]=p.detach().clone(); p.add_(v,alpha=float(scale))
    try: yield
    finally:
        with torch.no_grad():
            for name,value in saved.items(): named[name].copy_(value)

@torch.no_grad()
def residual_streams(model,ctx):
    ids,mask,_=ctx[0]
    return [h.detach().float().cpu() for h in model.model(input_ids=ids,attention_mask=mask,use_cache=False,output_hidden_states=True).hidden_states]

def first_updates(model,lr=BASE_LR,eps=EPS,wd=FT_WD):
    return {name:(-lr*(p.grad.detach().float()/(p.grad.detach().float().abs()+eps)+wd*p.detach().float())) for name,p in model.named_parameters()}

@torch.no_grad()
def normalization_inputs(model,row):
    stats=[]; hooks=[]
    def record(name,module,args):
        x=args[0].detach().float()
        ratio=(x.square().mean(-1)/model.config.rms_norm_eps).flatten()
        values=torch.quantile(ratio,torch.tensor([.01,.5,.99],device=ratio.device)).tolist()
        stats.append(dict(module=name,layer=layer(name),family=name.split('.')[-1],q01=values[0],median=values[1],q99=values[2]))
    for name,module in model.named_modules():
        if module.__class__.__name__=='Olmo2RMSNorm':
            hooks.append(module.register_forward_pre_hook(lambda mod,args,k=name:record(k,mod,args)))
    ids,mask,_=collate([row]) # single unpadded sequence
    try: model.model(input_ids=ids,attention_mask=mask,use_cache=False)
    finally:
        for hook in hooks: hook.remove()
    return stats

sym_rows=[]; stream_rows=[]; norm_input_rows=[]; sym_gate={}
for recipient,donor in PAIRS:
    profile=profiles[recipient,donor]
    for dtype in ([torch.float32] if RUN_FP32_SYMMETRY else [])+([torch.bfloat16] if RUN_BF16_SENSITIVITY else []):
        model=load_model(recipient,dtype)
        ctx=make_context('math','cal'); rows=next(diagnostic_batches('math','cal'))
        z0=context_logits(model,ctx); h0=residual_streams(model,ctx)
        backward_batch(model,rows); update=first_updates(model,lr=best_lr['math',recipient])
        with perturbed(model,update): step_kl=kl(z0,context_logits(model,ctx))
        del update; model.zero_grad(set_to_none=True)
        cases={'repeat':{},'identity':{name:1. for name in profile},'full':{name:1/a for name,a in profile.items()}}
        for fam in ELIGIBLE: cases[fam]={name:1/a for name,a in profile.items() if family(name)==fam}
        cases['reciprocal']={'model.layers.0.mlp.up_proj.weight':.5,'model.layers.0.mlp.down_proj.weight':2.}
        cases['gate_negative']={'model.layers.0.mlp.gate_proj.weight':.5}
        for case,scale in cases.items():
            with scaled_weights(model,scale):
                z1=context_logits(model,ctx); h1=residual_streams(model,ctx)
                metrics=output_metrics(z0,z1)
                y=context_targets(ctx)
                metrics['ce_change']=(F.cross_entropy(z1,y)-F.cross_entropy(z0,y)).item()
                if case in ('repeat','full'):
                    for item in normalization_inputs(model,rows[0]):
                        norm_input_rows.append(dict(recipient=recipient,donor=donor,dtype=str(dtype),case=case,**item))
            threshold=min(SYMMETRY_KL_LIMIT,SYMMETRY_STEP_FRACTION*max(step_kl,0.))
            sym_rows.append(dict(recipient=recipient,donor=donor,dtype=str(dtype),case=case,step_kl=step_kl,kl_threshold=threshold,pass_kl=metrics['kl']<=threshold,**metrics))
            for i,(x,y) in enumerate(zip(h0,h1)):
                stream_rows.append(dict(recipient=recipient,donor=donor,dtype=str(dtype),case=case,layer=i,relative_error=((x-y).norm()/x.norm()).item()))
        del model,h0,h1,z0,z1; cleanup()
sym=save_table(pd.DataFrame(sym_rows),'symmetry_validation')
streams=save_table(pd.DataFrame(stream_rows),'symmetry_stream_errors')
norm_inputs=save_table(pd.DataFrame(norm_input_rows),'rmsnorm_input_geometry')
display(sym)

In [ ]:
for (r,d),g in sym.groupby(['recipient','donor']):
    fig,ax=plt.subplots(figsize=(10,4))
    plot=g.assign(kl_plot=g.kl.clip(lower=1e-16))
    sns.barplot(data=plot,x='case',y='kl_plot',hue='dtype',ax=ax)
    ax.set(yscale='log',ylabel='Mean predictive KL (floor 1e-16)',title=f'{r:g} receives {d:g}'); ax.tick_params(axis='x',rotation=35); finish(fig,f'symmetry_{r:g}_{d:g}')
    fig,ax=plt.subplots(figsize=(10,4)); q=streams[(streams.recipient==r)&(streams.donor==d)&(streams.dtype=='torch.float32')]
    sns.heatmap(np.log10(q.pivot(index='case',columns='layer',values='relative_error').clip(lower=1e-12)),cmap='mako',ax=ax,cbar_kws={'label':'log10 relative error'},annot=False); finish(fig,f'symmetry_streams_{r:g}_{d:g}')
fig,ax=plt.subplots(figsize=(10,4)); q=norm_inputs[(norm_inputs.recipient==.1)&(norm_inputs.donor==1.)&(norm_inputs.dtype=='torch.float32')]
sns.boxplot(data=q,x='family',y='q01',hue='case',ax=ax); ax.set(yscale='log',ylabel='1st percentile of RMS input energy / epsilon'); ax.tick_params(axis='x',rotation=25); finish(fig,'rmsnorm_epsilon_regime')

## V. Inherited optimizer dynamics
[Algebra] Under exact scale invariance, $g(cW)=g(W)/c$. Fresh Adam gives $A(g)=g/(|g|+\epsilon)$; after scaling, $A(g/c)=g/(|g|+c\epsilon)$. The transported update coefficient is $c^{-1}(|g|+\epsilon)/(|g|+c\epsilon)$ for nonzero gradient coordinates. Angular displacement is $\operatorname{atan2}(\|\Delta w_\perp\|,r+u^\top\Delta w)$, not the unstable small-angle arccos. The ideal prediction $1/c$ is a check, not a discovery. Compare observed gradients against $g/c$ separately from finite-epsilon distortion. Under exact symmetry and matched optimizer histories, exact emulation uses $(\eta/c,c\epsilon,c\lambda)$ on the unscaled recipient; compensated physical transplant uses $(c\eta,\epsilon/c,\lambda/c)$ on the rescaled recipient. Fixed scale factors throughout a trajectory. No clipping in the primary clean experiment; sensitivity to clipping and decay is separate.

In [ ]:
def angular_stats(w,delta):
    w=w.detach().float(); delta=delta.detach().float()
    r=w.norm(); radial=(w.flatten()@delta.flatten())/r
    tangential_sq=(delta.square().sum()-radial.square()).clamp_min(0)
    tangential=tangential_sq.sqrt()
    return dict(norm=r.item(),relative_update=(delta.norm()/r).item(),radial=radial.item(),
                tangent=tangential.item(),angle=torch.atan2(tangential,r+radial).item(),first_order_angle=(tangential/r).item())

optimizer_rows=[]; grad_hist=[]
for recipient,donor in PAIRS:
    profile=profiles[recipient,donor]; scale={k:1/v for k,v in profile.items()}
    model=load_model(recipient); named=dict(model.named_parameters()); lr=best_lr['math',recipient]
    for task in TASKS:
        stream=diagnostic_batches(task,'cal')
        for batch_id in range(DIAG_REPEATS):
            rows=next(stream); backward_batch(model,rows)
            g0={k:p.grad.detach().clone() for k,p in named.items()}
            base={k:angular_stats(p,-lr*g0[k]/(g0[k].abs()+EPS)) for k,p in named.items() if candidate(k)}
            with scaled_weights(model,scale):
                backward_batch(model,rows)
                for k,p in named.items():
                    if not candidate(k): continue
                    g=g0[k]; gp=p.grad.detach(); c=scale[k]; a=g/(g.abs()+EPS); ap=gp/(gp.abs()+EPS)
                    theoretical=g/(g.abs()+c*EPS)
                    ratio=(g.abs()/EPS).flatten(); nz=ratio[ratio>0]
                    logbins=torch.histc(nz.log10().clamp(-12,12),bins=96,min=-12,max=12).cpu().numpy()
                    grad_hist.append(dict(recipient=recipient,donor=donor,task=task,batch=batch_id,family=family(k),zero_fraction=(ratio==0).float().mean().item(),counts=logbins))
                    obs=angular_stats(p,-lr*ap)
                    denominator=base[k]['angle']
                    optimizer_rows.append(dict(recipient=recipient,donor=donor,task=task,batch=batch_id,parameter=k,layer=layer(k),family=family(k),
                        predicted=1/c,observed=obs['angle']/denominator if denominator>0 else np.nan,
                        gradient_error=((c*gp-g).norm()/g.norm()).item(),epsilon_error=((theoretical-a).norm()/a.norm()).item(),
                        actual_adaptive_error=((ap-theoretical).norm()/theoretical.norm()).item(),small_gradient_fraction=(ratio<10).float().mean().item(),
                        **{f'baseline_{x}':v for x,v in base[k].items()},**{f'transplant_{x}':v for x,v in obs.items()}))
            del g0; model.zero_grad(set_to_none=True)
    del model,named; cleanup()
opt_geometry=save_table(pd.DataFrame(optimizer_rows),'optimizer_step_geometry')
for (r,d),g in opt_geometry.groupby(['recipient','donor']):
    for fam,fg in g.groupby('family'):
        q=sym[(sym.recipient==r)&(sym.donor==d)&(sym.dtype=='torch.float32')&(sym.case==fam)]
        sym_gate[r,d,fam]=bool(q.pass_kl.all() and fg.gradient_error.quantile(.95)<=SYMMETRY_GRAD_LIMIT)
family_gate=save_table(pd.DataFrame([dict(recipient=r,donor=d,family=f,eligible_validated=v) for (r,d,f),v in sym_gate.items()]),'symmetry_family_gate')
display(family_gate)

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(12,10))
for ax,((r,d),g) in zip(axes.flat,opt_geometry.groupby(['recipient','donor'])):
    sns.scatterplot(data=g,x='predicted',y='observed',hue='family',alpha=.3,s=15,ax=ax,legend=ax is axes[0,0])
    l=min(g.predicted.min(),g.observed.min()); h=max(g.predicted.max(),g.observed.max()); ax.plot([l,h],[l,h],'k--'); ax.set(title=f'{r:g} receives {d:g}')
finish(fig,'angular_amplification')
fig,ax=plt.subplots(figsize=(9,4))
for fam in ELIGIBLE:
    hist=np.sum([r['counts'] for r in grad_hist if r['family']==fam],axis=0); ax.plot(np.linspace(-12,12,97)[1:],hist.cumsum()/hist.sum(),label=fam)
ax.axvline(1,ls='--',color='k'); ax.set(xlabel='log10(|g| / Adam epsilon), nonzero coordinates',ylabel='ECDF (binned)'); ax.legend(); finish(fig,'adam_epsilon_ecdf')
fig,ax=plt.subplots(1,3,figsize=(16,4))
g=opt_geometry[(opt_geometry.recipient==.1)&(opt_geometry.donor==1.)]
for axis,key in zip(ax,['gradient_error','epsilon_error','actual_adaptive_error']):
    sns.heatmap(g.pivot_table(index='family',columns='layer',values=key).reindex(ELIGIBLE),ax=axis,cmap='mako',annot=False); axis.set_title(key)
finish(fig,'optimizer_errors')

### V.2 Physical transplant, exact emulation, compensated transplant, LR-only emulation
[Proposed] Run paired trajectories sequentially to avoid keeping five model instances live. A physical transplant is tested both on the full candidate profile (even if it fails, as a diagnostic) and, for inference, on validated families only. Compare predictive KL at every checkpoint, final transported parameter error, and actual held-out loss. A family-level gate does not certify the combined intervention; the full-profile KL and identity tests are also required. Fresh Adam states are reset per run. Repeat with nonzero WD and global clipping as a deliberate symmetry-breaking sensitivity condition.

In [ ]:
def identity_run(recipient,profile,task,arm,lr,wd=0.,clip=None):
    model=load_model(recipient); scales={k:1/a for k,a in profile.items()}
    physical=arm in ('physical','compensated')
    if physical:
        with torch.no_grad():
            for name,p in model.named_parameters(): p.mul_(scales[name])
    if arm=='exact': opt=optimizer(model,lr,profile,'exact',wd=wd)
    elif arm=='compensated': opt=optimizer(model,lr,scales,'exact',wd=wd)
    elif arm=='lr_only': opt=optimizer(model,lr,profile,'lr_only',wd=wd)
    else: opt=optimizer(model,lr,wd=wd)
    stream=training_batches(task,DIAG_BATCH,DATA_SEED); ctx=make_context(task,'cal')
    result={}
    for t in range(IDENTITY_STEPS+1):
        if t in (0,1,IDENTITY_STEPS): result[t]=context_logits(model,ctx).cpu()
        if t==IDENTITY_STEPS: break
        backward_batch(model,next(stream))
        if clip is not None: torch.nn.utils.clip_grad_norm_(model.parameters(),clip)
        opt.step()
    transported={k:(p.detach().cpu()/scales[k] if physical else p.detach().cpu().clone()) for k,p in model.named_parameters()}
    del opt,model; cleanup()
    return result,transported

identity_rows=[]
for r,d in PAIRS:
    full=profiles[r,d]
    restricted={k:a if sym_gate.get((r,d,family(k)),False) else 1. for k,a in full.items()}
    for label,profile in [('full',full),('validated',restricted)]:
        for wd,clip in [(0.,None),(.1,None),(.1,1.)]:
            arms={arm:identity_run(r,profile,'math',arm,best_lr['math',r],wd,clip) for arm in ['baseline','physical','exact','compensated','lr_only']}
            for left,right in [('physical','exact'),('baseline','compensated'),('physical','lr_only')]:
                lz,lw=arms[left]; rz,rw=arms[right]
                error=math.sqrt(sum((lw[k]-rw[k]).double().square().sum().item() for k in lw)/sum(lw[k].double().square().sum().item() for k in lw))
                y=context_targets(make_context('math','cal')).cpu()
                for t in lz:
                    metrics=output_metrics(lz[t],rz[t]); drift=kl(lz[0],lz[t])
                    # At t=0, use the absolute numerical cap; later use both absolute and step-relative caps.
                    tol=SYMMETRY_KL_LIMIT if t==0 else min(SYMMETRY_KL_LIMIT,SYMMETRY_STEP_FRACTION*max(drift,0.))
                    identity_rows.append(dict(recipient=r,donor=d,profile=label,wd=wd,clip=clip,left=left,right=right,step=t,final_transport_error=error,
                        context_loss_difference=(F.cross_entropy(lz[t],y)-F.cross_entropy(rz[t],y)).item(),identity_kl_limit=tol,
                        pass_identity=metrics['kl']<=tol and error<=IDENTITY_PARAMETER_TOL,**metrics))
            del arms; cleanup()
identity=save_table(pd.DataFrame(identity_rows),'optimizer_identity_tests'); display(identity)
fig,ax=plt.subplots(figsize=(10,4))
g=identity[(identity.recipient==.1)&(identity.donor==1.)&(identity.profile=='full')&(identity.wd==0)].copy(); g['pair']=g.left+' / '+g.right; g['kl_plot']=g.kl.clip(lower=1e-16)
sns.lineplot(data=g,x='step',y='kl_plot',hue='pair',marker='o',ax=ax); ax.set(yscale='log',ylabel='Predictive KL (floor 1e-16)'); finish(fig,'optimizer_identity')

## VI. Fisher functional adaptation geometry — core
[Algebra] For the actual next optimizer update $v$, define $d_i=J_i v_i$ and $D=[d_1,\ldots,d_m]$. The baseline is $D\mathbf 1$; an LR-only profile $a$ produces $Da$ to first order. For categorical probabilities $p$, $F_z=\operatorname{diag}(p)-pp^\top$. We compute the inner product by the transform $\tilde d=\sqrt{p}(d-\mathbb E_p d)$, then $G=\tilde D^\top\tilde D/n$ over sampled token contexts. No full Jacobian or Fisher is constructed. Centered-logit Gram is a sensitivity analysis. The Gram is task/optimizer/context-dependent, not the full functional tangent space.

[Proposed] For exact-emulation updates with changed epsilon/WD, use a second contribution basis $D_E$; the augmented Gram of $[D,D_E]$ retains cross terms. This avoids assuming that the exact emulation is a scalar multiple within each block. Primary hierarchy analysis is for LR-only $Da$; report discrepancy from exact emulation. All ineligible families are included. The numerical derivative is a central finite difference with convergence checks on aggregate directions and representative blocks. The scale $h$ multiplies the actual small optimizer update (not a unit vector). We use eager FP32 and repeat in a range of $h$; meaningful conclusions require stable results and agreement with finite-step predictive KL. Contexts are sampled separately for calibration and evaluation.

In [ ]:
def block_vector(groups,profile):
    return np.array([profile[names[0]] for names in groups.values()],dtype=np.float64)

def expand_profile(groups,values):
    return {name:float(a) for a,names in zip(values,groups.values()) for name in names}

def weighted_updates(updates,profile):
    return {k:v*profile[k] for k,v in updates.items()}

def adaptive_updates(model,lr,profile=None,mode='lr_only',opt=None):
    # Actual next AdamW step from the current gradient + moments; no mutation of state.
    # With opt=None, moments are zero. With opt set, its existing settings are used.
    output={}
    for i,(k,p) in enumerate(model.named_parameters()):
        a=1. if profile is None else profile[k]
        eps=EPS/a if mode=='exact' else EPS
        wd=FT_WD/a if mode=='exact' else FT_WD
        g=p.grad.detach().float()
        if opt is None:
            direction=g/(g.abs()+eps)
        else:
            pg=opt.param_groups[i]; state=opt.state[p]; b1,b2=pg['betas']
            t=int(state['step'].item())+1 if state else 1
            m=b1*state['exp_avg']+(1-b1)*g if state else (1-b1)*g
            v=b2*state['exp_avg_sq']+(1-b2)*g.square() if state else (1-b2)*g.square()
            direction=(m/(1-b1**t))/((v/(1-b2**t)).sqrt()+pg['eps'])
            eps=pg['eps']; wd=pg['weight_decay']; a=pg['lr']/lr
        output[k]=-lr*a*(direction+wd*p.detach().float())
    return output

@torch.no_grad()
def derivative(model,ctx,updates,h=FD_H):
    with perturbed(model,updates,h): plus=context_logits(model,ctx)
    with perturbed(model,updates,-h): minus=context_logits(model,ctx)
    return (plus-minus)/(2*h)

def fisher_transform(z,d):
    p=z.double().softmax(-1)
    d=d.double()
    return (p.sqrt()*(d-(p*d).sum(-1,keepdim=True))).flatten()/math.sqrt(z.shape[0])

def centered_transform(d): return centered(d.double()).flatten()/math.sqrt(d.shape[0])

def utility_dot(z,y,d):
    p=z.double().softmax(-1); d=d.double()
    return (d.gather(1,y[:,None]).squeeze(1)-(p*d).sum(-1)).mean().item()

def context_slice(ctx,start,end): return ctx[0],ctx[1][start:end]

def geometry(model,ctx,groups,base,exact=None,check_closure=False):
    # Evaluate each full-vocabulary derivative once. Temporary disk-backed raw
    # directions let us accumulate exact (unsketched) Gram products by token chunk.
    m=len(groups); width=m if exact is None else 2*m
    z=context_logits(model,ctx); y=context_targets(ctx); n,vocab=z.shape
    G=np.zeros((width,width)); C=np.zeros_like(G); b=np.zeros(width)
    with tempfile.TemporaryDirectory(prefix='gram_',dir=OUT) as directory:
        raw=np.memmap(Path(directory)/'directions.dat',mode='w+',dtype=np.float32,shape=(width,n,vocab))
        i=0
        for updates in ([base] if exact is None else [base,exact]):
            for names in groups.values():
                d=derivative(model,ctx,{k:updates[k] for k in names})
                raw[i]=d.cpu().numpy(); b[i]=utility_dot(z,y,d); i+=1
        raw.flush()
        for start in range(0,n,4):
            end=min(start+4,n)
            d=torch.from_numpy(np.array(raw[:,start:end,:])).to(DEVICE,dtype=torch.float64)
            p=z[start:end].double().softmax(-1)
            f=(p.sqrt()*(d-(p*d).sum(-1,keepdim=True))).flatten(1)
            c=(d-d.mean(-1,keepdim=True)).flatten(1)
            G+=(f@f.T).cpu().numpy()/n
            C+=(c@c.T).cpu().numpy()/n
            del d,f,c
        closure=np.nan
        if check_closure:
            summed=torch.from_numpy(np.sum(raw[:m],axis=0,dtype=np.float64)).to(DEVICE)
            direct=derivative(model,ctx,base)
            target=fisher_transform(z,direct)
            closure=(fisher_transform(z,summed-direct).norm()/target.norm()).item() if target.norm()>0 else np.nan
        del raw
    return G,C,b,closure

def norm_G(G,a): return math.sqrt(max(float(a@G@a),0.))

def fit_global(G,a):
    one=np.ones(len(a)); base=one@G@one; target=a@G@a
    if base<=0 or target<=0:
        return dict(k=np.nan,amplification=np.nan,cosine=np.nan,residual=np.nan,residual_unconstrained=np.nan,kappa=np.nan,chi=np.nan),np.full_like(G,np.nan)
    cross=one@G@a; k0=cross/base; k=max(0.,k0)
    Q=G-np.outer(G@one,G@one)/base; Q=(Q+Q.T)/2
    e=a-k*one; eig=np.linalg.eigvalsh(Q)
    return dict(k=k,amplification=math.sqrt(target/base),cosine=np.clip(cross/math.sqrt(base*target),-1,1),
                residual=norm_G(G,e)/math.sqrt(target),residual_unconstrained=norm_G(Q,a)/math.sqrt(target),
                kappa=base/np.trace(G),chi=math.sqrt(max(eig[-1],0.)/base)),Q

def generalized_fit(G,target,design):
    # Minimize ||G^(1/2) (target - B beta)|| with beta >= 0, including singular G.
    eig,V=np.linalg.eigh((G+G.T)/2)
    A=np.sqrt(eig.clip(min=0))[:,None]*V.T
    beta,_=nnls(A@design,A@target,maxiter=100*design.shape[1])
    return beta

def residual_for(G,a,x):
    denominator=norm_G(G,a)
    return norm_G(G,a-x)/denominator if denominator>0 else np.nan

def summarize_basis(G,a,b):
    metrics,Q=fit_global(G,a); one=np.ones(len(a)); k=metrics['k']; R=metrics['amplification']
    metrics.update(residual_utility=float(b@(a-k*one)),matched_global_utility=float(b@(a-R*one)),
                   residual_efficiency=float(b@(a-k*one))/norm_G(G,a-k*one) if norm_G(G,a-k*one)>0 else np.nan)
    return metrics,Q

### VI.1 Numerical derivative validation
[Proposed] Select aggregate baseline, norm-weighted, and representative individual block directions. Compare centered-logit/Fisher derivatives over a predeclared scale grid; record finite-step KL versus the quadratic prediction. A substantial discrepancy is numerical or nonlinear evidence, not proof that the hypothesis fails. The default derivative scale is fixed before evaluation; use this calibration to revise it and rerun all subsequent sections if necessary. An optional autodiff JVP on a small analytic model verifies the estimator independently.

In [ ]:
# Analytic smooth model: derivative is known, so this is a scientific numerical check, not a shape assertion.
x=torch.tensor([.3,.7],dtype=torch.float64); v=torch.tensor([.02,-.01],dtype=torch.float64)
fun=lambda q: torch.stack((q.sum(),q[1].square(),q[0].sin()))
_,jvp=torch.func.jvp(fun,(x,),(v,))
display(pd.DataFrame([dict(h=h,absolute_error=((fun(x+h*v)-fun(x-h*v))/(2*h)-jvp).norm().item()) for h in FD_SCALES]))
fd_rows=[]
for r,d in PAIRS:
    model=load_model(r); groups=groups_of(model); lr=best_lr['math',r]
    rows=next(diagnostic_batches('math','cal')); backward_batch(model,rows)
    base=adaptive_updates(model,lr); a=profiles[r,d]; weighted=weighted_updates(base,a); ctx=make_context('math','cal'); z=context_logits(model,ctx)
    candidates={'baseline':base,'weighted':weighted}
    for fam in ELIGIBLE:
        k=next(k for k in base if family(k)==fam); candidates[fam]={k:base[k]}
    for kind,update in candidates.items():
        ref=derivative(model,ctx,update,FD_H); rf=fisher_transform(z,ref)
        for h in FD_SCALES:
            current=derivative(model,ctx,update,h); cf=fisher_transform(z,current)
            with perturbed(model,update,h): actual=kl(z,context_logits(model,ctx))
            predicted=.5*h*h*cf.square().sum().item()
            fd_rows.append(dict(recipient=r,donor=d,kind=kind,h=h,fisher_norm=cf.norm().item(),
                 relative_to_reference=((cf-rf).norm()/rf.norm()).item() if rf.norm()>0 else np.nan,
                 actual_kl=actual,quadratic_kl=predicted,kl_ratio=actual/predicted if predicted>0 else np.nan))
    del model,base,weighted,candidates; cleanup()
fd=save_table(pd.DataFrame(fd_rows),'finite_difference_validation'); display(fd)
fig,ax=plt.subplots(1,2,figsize=(12,4)); sns.lineplot(data=fd,x='h',y='relative_to_reference',hue='kind',marker='o',ax=ax[0]); sns.lineplot(data=fd,x='h',y='kl_ratio',hue='kind',marker='o',ax=ax[1],legend=False)
for axis in ax: axis.set_xscale('log')
ax[1].axhline(1,ls='--',color='k'); finish(fig,'finite_difference_convergence')

### VI.2 Primary fine-grained Gram experiment
[Proposed] All 96 candidate matrices (16 layers × 6 families) are distinct coefficient coordinates. Noneligible tensors are grouped by family (each such group always receives multiplier 1 in the inherited profile). Compute both calibration and held-out Grams on distinct contexts and distinct gradient batches; within each split, all intervention arms share the same gradient. Geometry on retention uses exactly the same training update as downstream. The implementation computes each derivative once and temporarily stores raw directions on disk to accumulate the full Gram in token chunks. Save one NPZ per pair/task/repeat/split so a partial run can be inspected. Each repeat is a paired batch/context unit, not a collection of independent tokens or layers.

[Algebra] $k^*=\max(0,\mathbf1^\top Ga/(\mathbf1^\top G\mathbf1))$. $Q=G-G\mathbf1\mathbf1^\top G/(\mathbf1^\top G\mathbf1)$. For positive projection, $E_F^2=\delta^\top Q\delta/(a^\top Ga)$, $\delta=a-\bar a\mathbf1$. If the projection is negative, the constrained residual is 1. $\kappa=\mathbf1^\top G\mathbf1/\operatorname{tr}G$ diagnoses net cancellation, not guaranteed rotation. $\chi=\sqrt{\lambda_{\max}(Q)/(\mathbf1^\top G\mathbf1)}$ is the worst local angular sensitivity in chosen coefficient coordinates. $Q$ is positive semidefinite up to numerical error. Its eigenmodes describe the sampled optimizer directions, not universal neural features.

In [ ]:
GRAM_DIR=OUT/'grams'; GRAM_DIR.mkdir(exist_ok=True)
geometry_rows=[]; gram_records=[]; eigen_rows=[]
for r,d in PAIRS:
    model=load_model(r); groups=groups_of(model); labels=list(groups); m=len(labels)
    profile=profiles[r,d]; a=block_vector(groups,profile); one=np.ones(m)
    for task in TASKS:
        for split in ('cal','eval'):
            stream=diagnostic_batches(task,split)
            for rep in tqdm(range(DIAG_REPEATS),desc=f'{r:g}->{d:g} {task} {split}'):
                backward_batch(model,next(stream)); lr=best_lr[task,r]
                base=adaptive_updates(model,lr); exact=adaptive_updates(model,lr,profile,'exact')
                ctx=make_context(task,split,DATA_SEED+rep); ret=make_context('retention',split,DATA_SEED+rep)
                # Exact-emulation update differs internally, so use augmented basis on downstream.
                GD,CD,bD,closure=geometry(model,ctx,groups,base,exact,check_closure=True)
                GP,CP,bP,ret_closure=geometry(model,ret,groups,base,exact,check_closure=True)
                G=GD[:m,:m]; b=bD[:m]; metrics,Q=summarize_basis(G,a,b)
                cm,_=fit_global(CD[:m,:m],a)
                x=np.r_[a,np.zeros(m)]; y=np.r_[np.zeros(m),one]
                err=norm_G(GD,x-y)/norm_G(GD,y) if norm_G(GD,y)>0 else np.nan
                cross=float(x@GD@y); denom=norm_G(GD,x)*norm_G(GD,y)
                aug_base=np.r_[one,np.zeros(m)]
                base_energy=float(aug_base@GD@aug_base)
                exact_k=max(0.,float(aug_base@GD@y/base_energy)) if base_energy>0 else np.nan
                exact_residual=norm_G(GD,y-exact_k*aug_base)/norm_G(GD,y) if norm_G(GD,y)>0 else np.nan
                # Compare aggregate derivative with sum of block derivatives through Gram norm.
                # Direct aggregate finite difference has finite-h nonlinear error; report closure.
                record=dict(recipient=r,donor=d,task=task,split=split,repeat=rep,**metrics,centered_residual=cm['residual'],
                            exact_lr_only_error=err,exact_lr_only_cosine=cross/denom if denom>0 else np.nan,
                            exact_global_residual=exact_residual,aggregate_vector_closure=closure,retention_vector_closure=ret_closure,
                            min_gram_eigenvalue=np.linalg.eigvalsh(G)[0],q_null_error=np.linalg.norm(Q@one))
                geometry_rows.append(record)
                eig,V=np.linalg.eigh(Q); order=np.argsort(eig)[::-1]; eig=eig[order].clip(min=0); V=V[:,order]; delta=a-a.mean()
                contrib=eig*(V.T@delta)**2
                for i,(ev,energy) in enumerate(zip(eig,contrib)):
                    eigen_rows.append(dict(recipient=r,donor=d,task=task,split=split,repeat=rep,mode=i,eigenvalue=ev,residual_energy=energy))
                fname=f'wd{r:g}_donor{d:g}_{task}_{split}_{rep}.npz'
                np.savez_compressed(GRAM_DIR/fname,GD=GD,GP=GP,CD=CD,CP=CP,bD=bD,bP=bP,a=a,Q=Q,labels=np.array(labels),eigenvalues=eig,eigenvectors=V)
                gram_records.append(dict(recipient=r,donor=d,task=task,split=split,repeat=rep,file=fname))
                del base,exact,GD,GP,CD,CP; model.zero_grad(set_to_none=True); cleanup()
    del model; cleanup()
geometry_df=save_table(pd.DataFrame(geometry_rows),'functional_geometry')
gram_manifest=save_table(pd.DataFrame(gram_records),'gram_manifest')
eigen_df=save_table(pd.DataFrame(eigen_rows),'residual_eigenmodes')
display(geometry_df.groupby(['recipient','donor','task','split'])[['residual','cosine','amplification','kappa','chi','exact_lr_only_error','aggregate_vector_closure']].median())

In [ ]:
def load_gram(row):
    with np.load(GRAM_DIR/row.file) as f: return {k:f[k] for k in f.files}

def group_label(name):
    return name.split('.')[-1] if name.startswith('other.') else family(name)

def group_design(labels,kind):
    if kind=='global': names=['all']*len(labels)
    elif kind=='two_group': names=['other' if k.startswith('other.') else 'eligible' for k in labels]
    elif kind=='family': names=[group_label(k) for k in labels]
    elif kind=='blockwise': names=list(labels)
    unique=list(dict.fromkeys(names))
    return np.array([[float(n==u) for u in unique] for n in names]),unique

# Representative pair/task: plot raw/normalized Fisher Gram and Q, plus aggregate family interactions.
row=gram_manifest[(gram_manifest.recipient==.1)&(gram_manifest.donor==1.)&(gram_manifest.task=='math')&(gram_manifest.split=='eval')].iloc[0]
record=load_gram(row); labels=record['labels']; m=len(labels); G=record['GD'][:m,:m]; Q=record['Q']; a=record['a']
scale=np.sqrt(np.diag(G).clip(min=0)); C=np.divide(G,np.outer(scale,scale),out=np.full_like(G,np.nan),where=np.outer(scale,scale)>0)
fig,axes=plt.subplots(1,3,figsize=(20,6))
for ax,mat,title in zip(axes,[G,C,Q],['Fisher Gram G','Normalized alignment','Residual Gram Q']):
    lim=np.nanmax(np.abs(mat)); sns.heatmap(mat,cmap='vlag',center=0,vmin=-lim,vmax=lim,xticklabels=False,yticklabels=False,annot=False,ax=ax); ax.set_title(title)
finish(fig,'fine_grained_gram')
B,families=group_design(labels,'family'); aggregate=B.T@G@B; s=np.sqrt(np.diag(aggregate).clip(min=0)); corr=np.divide(aggregate,np.outer(s,s),out=np.full_like(aggregate,np.nan),where=np.outer(s,s)>0)
fig,ax=plt.subplots(figsize=(9,7)); sns.heatmap(corr,xticklabels=families,yticklabels=families,cmap='vlag',center=0,vmin=-1,vmax=1,annot=False,ax=ax); finish(fig,'family_alignment')
eig=record['eigenvalues']; V=record['eigenvectors']; energy=eig*(V.T@(a-a.mean()))**2
fig,ax=plt.subplots(1,2,figsize=(13,4)); ax[0].plot(eig/eig.sum(),label='Q eigenvalues'); ge=np.linalg.eigvalsh(G)[::-1].clip(min=0); ax[0].plot(ge/ge.sum(),label='G eigenvalues'); ax[0].set(yscale='log',xlabel='Mode',ylabel='Normalized eigenvalue'); ax[0].legend(); ax[1].bar(np.arange(min(20,len(eig))),energy[:20]/max(energy.sum(),1e-30)); ax[1].set(xlabel='Q eigenmode',ylabel='Fraction of residual energy'); finish(fig,'gram_eigenspectra')
fig,ax=plt.subplots(figsize=(12,5)); sns.heatmap(V[:,:8].T,cmap='vlag',center=0,xticklabels=False,yticklabels=np.arange(8),annot=False,ax=ax); ax.set(xlabel='Parameter block (stored order)',ylabel='Q eigenmode'); finish(fig,'residual_mode_loadings')
fig,ax=plt.subplots(1,2,figsize=(12,4)); held=geometry_df[geometry_df.split=='eval']; sns.scatterplot(data=held,x='kappa',y='residual',hue='task',ax=ax[0]); sns.scatterplot(data=held,x='chi',y='residual',hue='task',ax=ax[1]); finish(fig,'cancellation_sensitivity')

## VII. Hierarchical compensation — how much structure is needed?
[Algebra] Fit $\beta\ge0$ to minimize $(a-B\beta)^\top G(a-B\beta)$ for global, two-group, family, and full blockwise designs. Nested classes have nonincreasing *in-sample* residual; full blockwise target is reproduced by construction. Fit on the sum of calibration Grams (normalized per context unit), then freeze coefficients for independent gradient batches and contexts. The family model includes separate noneligible families, which refines two-group. The blockwise arm is the measured profile, not a fitted model. Shuffling the eligible ratios gives conditional references, not valid randomization p-values absent exchangeability. We include both global shuffles and within-family shuffles, each preserving the corresponding marginal distribution.

In [ ]:
fits={}; hierarchy_rows=[]; shuffle_rows=[]; profile_store={}
for (r,d,task),g in gram_manifest.groupby(['recipient','donor','task']):
    cal_recs=[load_gram(row) for row in g[g.split=='cal'].itertuples(index=False)]
    a=cal_recs[0]['a']; labels=cal_recs[0]['labels']; m=len(a); one=np.ones(m)
    Gcal=np.mean([x['GD'][:m,:m] for x in cal_recs],axis=0)
    g_profiles={'baseline':one,'measured':a}
    for kind in ['global','two_group','family']:
        B,groupnames=group_design(labels,kind); beta=generalized_fit(Gcal,a,B); g_profiles[kind]=B@beta
        fits[r,d,task,kind]=dict(zip(groupnames,beta.tolist()))
    eligible=np.array([not name.startswith('other.') for name in labels]); mask=eligible.astype(float)
    g_profiles['selective']=one+mask*(np.exp(np.log(a[eligible]).mean())-1)
    rng=np.random.default_rng(DATA_SEED)
    permutations=[]
    for i in range(N_SHUFFLES):
        perm=a.copy(); perm[eligible]=rng.permutation(a[eligible]); permutations.append(('all',i,perm))
        perm=a.copy()
        for fam in ELIGIBLE:
            ids=np.array([group_label(k)==fam for k in labels]); perm[ids]=rng.permutation(a[ids])
        permutations.append(('within_family',i,perm))
    profile_store[r,d,task]=(labels,g_profiles,permutations)
    for row in g.itertuples(index=False):
        x=load_gram(row); G=x['GD'][:m,:m]; GP=x['GP'][:m,:m]; b=x['bD'][:m]
        for kind,coeff in g_profiles.items():
            hierarchy_rows.append(dict(recipient=r,donor=d,task=task,split=row.split,repeat=row.repeat,profile=kind,
                 residual=residual_for(G,a,coeff),fisher_movement=norm_G(G,coeff),utility=float(b@coeff)))
        for kind,i,perm in permutations:
            metrics,_=fit_global(G,perm); dn=norm_G(G,perm); pn=norm_G(GP,perm)
            shuffle_rows.append(dict(recipient=r,donor=d,task=task,split=row.split,repeat=row.repeat,shuffle=kind,index=i,
                 residual=metrics['residual'],eff_D=float(b@perm)/dn if dn>0 else np.nan,eff_P=float(b@perm)/pn if pn>0 else np.nan))
(OUT/'hierarchy_coefficients.json').write_text(json.dumps([dict(recipient=r,donor=d,task=t,profile=k,coefficients=v) for (r,d,t,k),v in fits.items()],indent=2))
hierarchy=save_table(pd.DataFrame(hierarchy_rows),'compensation_hierarchy'); shuffles=save_table(pd.DataFrame(shuffle_rows),'profile_randomization')
fig,ax=plt.subplots(figsize=(11,5)); subset=hierarchy[(hierarchy.recipient==.1)&(hierarchy.donor==1.)&(hierarchy.task=='math')]
sns.boxplot(data=subset,x='profile',y='residual',hue='split',order=['global','selective','two_group','family','measured'],ax=ax); ax.axhline(EQUIV_RESIDUAL,ls='--',color='k'); finish(fig,'compensation_hierarchy')
fig,ax=plt.subplots(figsize=(9,4)); s=shuffles[(shuffles.recipient==.1)&(shuffles.donor==1.)&(shuffles.task=='math')&(shuffles.split=='eval')]; sns.histplot(data=s.groupby(['shuffle','index']).residual.mean().reset_index(),x='residual',hue='shuffle',element='step',stat='density',common_norm=False,ax=ax); ax.axvline(held[(held.recipient==.1)&(held.donor==1.)&(held.task=='math')].residual.mean(),color='k',ls='--',label='measured'); ax.legend(); finish(fig,'shuffled_residual_reference')

## VIII. Useful adaptation and retention — core
[Algebra] $b_i=-g_D^\top d_i$ is held-out loss reduction per unit coefficient, $U(a)=b^\top a$. Residual utility $b^\top(a-k^*\mathbf1)$ is not the same as advantage at equal Fisher movement. Compare $\mathcal E_D=b^\top a/\sqrt{a^\top G_Da}$ and $\mathcal E_P=b^\top a/\sqrt{a^\top G_Pa}$; report near-zero denominators as undefined. The latter measures predictive disruption, not forgetting; measure actual retention CE change as well. The local Fisher approximation is verified using finite steps, not assumed to remain true at realistic LR. Use calibration Grams to choose movement-matched step scales, then apply those scales unchanged to evaluation contexts. Report realized movement and compare curves only on common support. The gap-relative margin is in nats/token and applies to improvement at the anchor horizon, not a single-step Fisher residual.

In [ ]:
def local_metrics(G,GP,b,a):
    dn=norm_G(G,a); pn=norm_G(GP,a); utility=float(b@a)
    return dict(utility=utility,fisher_D=dn,fisher_P=pn,eff_D=utility/dn if dn>1e-14 else np.nan,
                eff_P=utility/pn if pn>1e-14 else np.nan)

utility_rows=[]; shuffle_percentiles=[]
for row in gram_manifest.itertuples(index=False):
    x=load_gram(row); a=x['a']; m=len(a); labels,g_profiles,permutations=profile_store[row.recipient,row.donor,row.task]
    G=x['GD'][:m,:m]; GP=x['GP'][:m,:m]; b=x['bD'][:m]; one=np.ones(m)
    base=local_metrics(G,GP,b,one); measured=local_metrics(G,GP,b,a)
    for kind,coef in g_profiles.items():
        metrics=local_metrics(G,GP,b,coef)
        metrics['matched_D_advantage']=measured['fisher_D']*(metrics['eff_D']-base['eff_D'])
        metrics['matched_P_advantage']=measured['fisher_P']*(metrics['eff_P']-base['eff_P'])
        utility_rows.append(dict(recipient=row.recipient,donor=row.donor,task=row.task,split=row.split,repeat=row.repeat,profile=kind,**metrics))
    exact=local_metrics(x['GD'],x['GP'],x['bD'],np.r_[np.zeros(m),one]); exact['matched_D_advantage']=measured['fisher_D']*(exact['eff_D']-base['eff_D']); exact['matched_P_advantage']=measured['fisher_P']*(exact['eff_P']-base['eff_P'])
    utility_rows.append(dict(recipient=row.recipient,donor=row.donor,task=row.task,split=row.split,repeat=row.repeat,profile='exact',**exact))
    for kind in ('all','within_family'):
        refs=[local_metrics(G,GP,b,p) for typ,_,p in permutations if typ==kind]
        shuffle_percentiles.append(dict(recipient=row.recipient,donor=row.donor,task=row.task,split=row.split,repeat=row.repeat,shuffle=kind,
             eff_D_percentile=np.mean([measured['eff_D']>=q['eff_D'] for q in refs]),eff_P_percentile=np.mean([measured['eff_P']>=q['eff_P'] for q in refs])))
utilities=save_table(pd.DataFrame(utility_rows),'local_linear_utility'); percentiles=save_table(pd.DataFrame(shuffle_percentiles),'shuffle_utility_percentiles')
fig,ax=plt.subplots(1,2,figsize=(14,5)); sub=utilities[(utilities.recipient==.1)&(utilities.donor==1.)&(utilities.task=='math')&(utilities.split=='eval')]
for axis,col in zip(ax,['eff_D','eff_P']):
    sns.boxplot(data=sub,x='profile',y=col,ax=axis); sns.stripplot(data=sub,x='profile',y=col,ax=axis,color='black',alpha=.45); axis.tick_params(axis='x',rotation=30)
finish(fig,'local_efficiency')

In [ ]:
# Matched finite-step comparisons. Use calibration Gram on the same recipient/task to set
# a global factor for each profile; never select scale on final held-out loss.
finite_rows=[]
for (r,d,task),g in gram_manifest.groupby(['recipient','donor','task']):
    labels,g_profiles,permutations=profile_store[r,d,task]; m=len(labels)
    g_cal=[load_gram(row) for row in g[g.split=='cal'].itertuples(index=False)]
    GD=np.mean([x['GD'] for x in g_cal],axis=0); GP=np.mean([x['GP'] for x in g_cal],axis=0)
    measured=np.r_[g_profiles['measured'],np.zeros(m)]
    # A fixed set of shuffled controls is evaluated with actual finite steps, not only Gram predictions.
    coeffs={kind:np.r_[a,np.zeros(m)] for kind,a in g_profiles.items()}
    coeffs.update({f'shuffle_{typ}_{i}':np.r_[a,np.zeros(m)] for typ,i,a in permutations if i<3})
    coeffs['exact']=np.r_[np.zeros(m),np.ones(m)]
    model=load_model(r); groups=groups_of(model); profile=profiles[r,d]; lr=best_lr[task,r]
    stream=diagnostic_batches(task,'eval')
    for rep in tqdm(range(DIAG_REPEATS),desc=f'Finite utility {r:g}->{d:g} {task}'):
        backward_batch(model,next(stream)); base=adaptive_updates(model,lr); exact=adaptive_updates(model,lr,profile,'exact')
        ctx=make_context(task,'eval',DATA_SEED+rep); ret=make_context('retention','eval',DATA_SEED+rep)
        z0=context_logits(model,ctx); p0=context_logits(model,ret)
        yd=context_targets(ctx); yp=context_targets(ret)
        loss0=F.cross_entropy(z0,yd).item(); retention0=F.cross_entropy(p0,yp).item()
        for kind,coef in coeffs.items():
            update=exact if kind=='exact' else weighted_updates(base,expand_profile(groups,coef[:m]))
            for budget,G in [('downstream',GD),('retention',GP)]:
                n=norm_G(G,coef); target=norm_G(G,measured)
                if n<=1e-14: continue
                match=target/n
                for s in STEP_SCALES:
                    with perturbed(model,update,match*s):
                        z1=context_logits(model,ctx); p1=context_logits(model,ret)
                    finite_rows.append(dict(recipient=r,donor=d,task=task,repeat=rep,profile=kind,budget=budget,step_scale=s,lr_factor=match*s,
                        downstream_improvement=loss0-F.cross_entropy(z1,yd).item(),retention_loss_change=F.cross_entropy(p1,yp).item()-retention0,
                        downstream_kl=kl(z0,z1),retention_kl=kl(p0,p1),predicted_cal_movement=target*s))
            if kind!='exact': del update
        del base,exact; model.zero_grad(set_to_none=True); cleanup()
    del model; cleanup()
finite=save_table(pd.DataFrame(finite_rows),'local_adaptation_retention')
fig,ax=plt.subplots(1,2,figsize=(14,5)); s=finite[(finite.recipient==.1)&(finite.donor==1.)&(finite.task=='math')&finite.profile.isin(['baseline','global','two_group','family','measured','exact','shuffle_all_0'])]
for axis,budget,xcol in zip(ax,['downstream','retention'],['downstream_kl','retention_kl']):
    v=s[s.budget==budget].groupby(['profile','step_scale'])[[xcol,'downstream_improvement']].mean().reset_index()
    sns.lineplot(data=v,x=xcol,y='downstream_improvement',hue='profile',marker='o',ax=axis); axis.set_title('Realized '+budget+' movement')
finish(fig,'finite_adaptation_retention_tradeoff')

# Compare actual finite utility at the predeclared KL budget derived from the calibration
# movement at scale 1. Linear interpolation only within observed monotone support.
matched_rows=[]
for keys,g in finite.groupby(['recipient','donor','task','repeat','budget','profile']):
    r,d,task,rep,budget,kind=keys
    xcol='downstream_kl' if budget=='downstream' else 'retention_kl'
    g=g.sort_values('step_scale'); target=.5*(g.predicted_cal_movement.iloc[0]/g.step_scale.iloc[0])**2
    x=np.r_[0.,g[xcol].to_numpy()]; y=np.r_[0.,g.downstream_improvement.to_numpy()]
    ret=np.r_[0.,g.retention_loss_change.to_numpy()]
    supported=np.all(np.diff(x)>0) and 0<=target<=x[-1]
    matched_rows.append(dict(recipient=r,donor=d,task=task,repeat=rep,budget=budget,profile=kind,target_kl=target,
        supported=bool(supported),matched_improvement=np.interp(target,x,y) if supported else np.nan,
        matched_retention_loss=np.interp(target,x,ret) if supported else np.nan))
matched_finite=save_table(pd.DataFrame(matched_rows),'finite_matched_utility')
display(matched_finite.groupby(['recipient','donor','task','budget','profile']).agg(support=('supported','mean'),mean_improvement=('matched_improvement','mean')))

## IX. Short trajectories — bidirectional and optimizer-state dependence
[Proposed] Repeat fixed LR-only, exact-emulation, global/two-group/family and shuffled profiles from each recipient (L, M, H), paired with the same training sequence and seed. Run to the native horizon and assess validation loss improvement, absolute loss and retention. The baseline uses the recipient's calibrated LR, not a poorly tuned arbitrary baseline. Global/group controls additionally calibrate a common scalar LR on the calibration partition; this avoids claiming advantage merely because their overall step scale was not tuned. The measured profile is also scalar-calibrated for a fair comparison. Fit all scalar controls on the same grid, including a global baseline; no test selection. Keep a fixed profile throughout training; dynamic norm matching is not the same intervention. At selected steps measure the next update using the actual Adam moment state and fixed held-out contexts. A local Gram of the current next-step direction characterizes reweighting *that* direction, not a counterfactual optimizer history. We also measure directional drift from each arm's own initial function; independently trained recipients are not treated as equivalent.

In [ ]:
def trajectory_run(r,d,task,kind,profile,lr,seed,split,points,track_geometry=False):
    seed_all(seed); model=load_model(r); opt=optimizer(model,lr,profile,'exact' if kind=='exact' else 'lr_only')
    stream=training_batches(task,TRAIN_BATCH,seed); ctx=make_context(task,split); ret=make_context('retention',split)
    z0=context_logits(model,ctx); p0=context_logits(model,ret)
    initial=eval_loss(model,task,split)[0]; ret0=eval_loss(model,'retention',split)[0]
    result=[]; dynamic=[]; tokens=supervised=0
    for t in range(max(points)+1):
        if t in points:
            loss,acc=eval_loss(model,task,split); retention=eval_loss(model,'retention',split)[0]
            result.append(dict(recipient=r,donor=d,task=task,profile=kind,lr=lr,seed=seed,split=split,step=t,tokens=tokens,supervised_tokens=supervised,
                loss=loss,improvement=initial-loss,accuracy=acc,retention_loss_change=retention-ret0,
                downstream_kl=kl(z0,context_logits(model,ctx)),retention_kl=kl(p0,context_logits(model,ret))))
        if t==max(points) and not track_geometry: break
        rows=next(stream); _,n,nt=backward_batch(model,rows)
        if track_geometry and t in (0,10,100,max(points)):
            groups=groups_of(model); update=adaptive_updates(model,lr,opt=opt)
            # Reweight the actual next direction by the donor profile on each state.
            a=block_vector(groups,profiles[r,d]); G,_,b,_=geometry(model,ctx,groups,update); GP,_,_,_=geometry(model,ret,groups,update)
            metrics,Q=fit_global(G,a); current=local_metrics(G,GP,b,np.ones(len(a))); hypothetical=local_metrics(G,GP,b,a)
            dynamic.append(dict(recipient=r,donor=d,task=task,profile=kind,seed=seed,step=t,**metrics,
                current_eff_D=current['eff_D'],current_eff_P=current['eff_P'],reweighted_eff_D=hypothetical['eff_D'],reweighted_eff_P=hypothetical['eff_P']))
            del update
        if t==max(points): break
        opt.step(); tokens+=nt; supervised+=n
    del opt,model; cleanup(); return result,dynamic

trajectory_rows=[]; trajectory_cal_rows=[]; dynamic_rows=[]
if RUN_TRAJECTORIES:
    for (r,d,task),(labels,group_profiles,perms) in profile_store.items():
        # group labels match model.named_parameters order as persisted in gram artifacts.
        model=load_model(r); groups=groups_of(model); del model; cleanup()
        arms={k:expand_profile(groups,v) for k,v in group_profiles.items() if k in ['baseline','global','two_group','family','measured']}
        arms['exact']=profiles[r,d]
        for typ,i,v in perms:
            if i==0: arms[f'shuffle_{typ}']=expand_profile(groups,v)
        selected={}
        for kind,profile in arms.items():
            # Equal tuning budget, multiplicative LR grid centered at recipient's native optimum.
            for factor in (.33,1.,3.):
                lr=best_lr[task,r]*factor
                rec,_=trajectory_run(r,d,task,kind,profile,lr,DATA_SEED,'cal',(0,ANCHOR_STEPS))
                trajectory_cal_rows.extend(rec)
            last=[x for x in trajectory_cal_rows if x['recipient']==r and x['donor']==d and x['task']==task and x['profile']==kind and x['step']==ANCHOR_STEPS]
            selected[kind]=min(last,key=lambda x:x['loss'])['lr']
        for kind,profile in arms.items():
            for seed in SEEDS:
                rec,dyn=trajectory_run(r,d,task,kind,profile,selected[kind],seed,'eval',TRAJECTORY_POINTS,track_geometry=(kind in ('baseline','measured','exact') and seed==SEEDS[0]))
                trajectory_rows.extend(rec); dynamic_rows.extend(dyn)
        save_table(pd.DataFrame(trajectory_rows),'short_trajectory_analysis')
        save_table(pd.DataFrame(trajectory_cal_rows),'trajectory_lr_calibration')
        save_table(pd.DataFrame(dynamic_rows),'trajectory_geometry')
    trajectories=pd.DataFrame(trajectory_rows); dynamic=pd.DataFrame(dynamic_rows)
    fig,ax=plt.subplots(1,2,figsize=(15,5)); s=trajectories[(trajectories.recipient==.1)&(trajectories.donor==1.)&(trajectories.task=='math')]
    sns.lineplot(data=s,x='step',y='improvement',hue='profile',marker='o',ax=ax[0],errorbar=None); sns.lineplot(data=s,x='step',y='retention_loss_change',hue='profile',marker='o',ax=ax[1],errorbar=None,legend=False); finish(fig,'short_trajectories')
    fig,ax=plt.subplots(figsize=(9,4)); s=dynamic[(dynamic.recipient==.1)&(dynamic.donor==1.)&(dynamic.task=='math')]; sns.lineplot(data=s,x='step',y='residual',hue='profile',marker='o',ax=ax); finish(fig,'trajectory_geometry')

## X. Evidence synthesis and decisions
[Proposed] The anchor is a *relevance gate*, not an instruction to suppress negative local results. If the native gap is absent/inconclusive, label the causal relevance to the paper as unestablished for this task/horizon. For each diagnostic use paired repeat/seed uncertainty, not individual tokens. With three training seeds the interval is only a descriptive conditional bootstrap, not precise population inference; show raw seed outcomes. Approximate equivalence needs an interval inside the declared margin; no significant difference is not equivalence. The local residual tolerance is a provisional protocol choice; if the finite-difference calibration or aggregate vector closure exceeds it, geometry decisions are marked inconclusive. The trajectory utility margin is gap-relative and calibrated independently. All conclusions remain conditional on the released checkpoints and local data/protocol. A large residual may be unhelpful, and matched movement is not synonymous with long-horizon plasticity.

In [ ]:
def decision_signed(low,high,m):
    if low>m: return 'positive meaningful difference'
    if high< -m: return 'negative meaningful difference'
    if low>=-m and high<=m: return 'approximate equivalence'
    return 'inconclusive'

def decision_residual(low,high,m):
    if low>m: return 'meaningful non-reproducibility'
    if high<=m: return 'approximate equivalence'
    return 'inconclusive'

ledger=[]
for (r,d,task),g in geometry_df[geometry_df.split=='eval'].groupby(['recipient','donor','task']):
    lo=min(r,d); ag=anchor_gate[(anchor_gate.task==task)&(anchor_gate.recipient==lo)&(anchor_gate.arm=='tuned_lr')].iloc[0]
    symmetry=sym[(sym.recipient==r)&(sym.donor==d)&(sym.case=='full')&(sym.dtype=='torch.float32')].pass_kl.all()
    validation=family_gate[(family_gate.recipient==r)&(family_gate.donor==d)].eligible_validated.all()
    joint=identity[(identity.recipient==r)&(identity.donor==d)&(identity.profile=='full')&(identity.wd==0)&(identity.left=='physical')&(identity.right=='exact')]
    joint_identity_kl=joint.kl.max()
    fd_subset=fd[(fd.recipient==r)&(fd.donor==d)&(fd.kind.isin(['baseline','weighted']))&(fd.h==.5)]
    numeric_error=max(fd_subset.relative_to_reference.max(),g.aggregate_vector_closure.max(),g.retention_vector_closure.max())
    common=dict(recipient=r,donor=d,task=task,anchor=ag.decision,fd_relative_error=numeric_error,full_symmetry=bool(symmetry),family_gradient_gate=bool(validation),max_identity_kl=joint_identity_kl,identity_validated=bool(joint.pass_identity.all()),
                interpretation_scope='conditional local checkpoint study; LR-only not automatically a gauge intervention')
    for metric,hypothesis in [('residual','H1a-local'),('exact_global_residual','H1a-local exact'),('exact_lr_only_error','LR-only vs exact')]:
        est,l,h=bootstrap_mean(g[metric]); ledger.append(dict(**common,hypothesis=hypothesis,metric=metric,estimate=est,ci_low=l,ci_high=h,margin=EQUIV_RESIDUAL,decision=decision_residual(l,h,EQUIV_RESIDUAL)))
    hg=hierarchy[(hierarchy.recipient==r)&(hierarchy.donor==d)&(hierarchy.task==task)&(hierarchy.split=='eval')]
    for kind in ('two_group','family'):
        est,l,h=bootstrap_mean(hg[hg.profile==kind].residual)
        ledger.append(dict(**common,hypothesis='H1b-'+kind,metric='held-out hierarchy residual',estimate=est,ci_low=l,ci_high=h,margin=EQUIV_RESIDUAL,decision=decision_residual(l,h,EQUIV_RESIDUAL)))
    ug=utilities[(utilities.recipient==r)&(utilities.donor==d)&(utilities.task==task)&(utilities.split=='eval')&(utilities.profile=='measured')]
    for metric in ('matched_D_advantage','matched_P_advantage'):
        est,l,h=bootstrap_mean(ug[metric]); ledger.append(dict(**common,hypothesis='H1b-utility local',metric=metric,estimate=est,ci_low=l,ci_high=h,margin=np.nan,
             decision='positive local signal' if l>0 else 'negative local signal' if h<0 else 'inconclusive; no single-step practical margin calibrated'))
    fg=matched_finite[(matched_finite.recipient==r)&(matched_finite.donor==d)&(matched_finite.task==task)]
    for budget,budget_group in fg.groupby('budget'):
        pivot=budget_group.pivot(index='repeat',columns='profile',values='matched_improvement')
        for control in ['baseline','two_group','family','shuffle_all_0','shuffle_within_family_0']:
            diff=(pivot['measured']-pivot[control]).dropna(); est,l,h=bootstrap_mean(diff)
            ledger.append(dict(**common,hypothesis='H1b-utility finite',metric=budget+' matched KL: measured minus '+control,estimate=est,ci_low=l,ci_high=h,margin=np.nan,
                decision='insufficient shared support' if len(diff)<DIAG_REPEATS//2 else 'positive local signal' if l>0 else 'negative local signal' if h<0 else 'inconclusive'))
    if RUN_TRAJECTORIES:
        tg=trajectories[(trajectories.recipient==r)&(trajectories.donor==d)&(trajectories.task==task)&(trajectories.step==ANCHOR_STEPS)]
        pivot=tg.pivot(index='seed',columns='profile',values='improvement')
        for control in ['baseline','global','two_group','family','shuffle_all','shuffle_within_family']:
            # Shuffled arms are the fixed first permutation of each type.
            est,l,h=bootstrap_mean(pivot['measured']-pivot[control]); m=margin[task,lo]
            ledger.append(dict(**common,hypothesis='H1b-trajectory',metric='measured minus '+control,estimate=est,ci_low=l,ci_high=h,margin=m,decision=decision_signed(l,h,m)))
evidence=pd.DataFrame(ledger)
noisy=(evidence.fd_relative_error>EQUIV_RESIDUAL)&evidence.hypothesis.str.contains('H1a-local|H1b-two_group|H1b-family|LR-only')
evidence.loc[noisy,'decision']='inconclusive: derivative calibration exceeds residual margin'
evidence=save_table(evidence,'evidence_ledger'); display(evidence)
summary_lines=['# Investigation A — evidence ledger','', 'Conditional on the released checkpoint revisions and the custom local protocol. No population-level causal mediation claim.','', '## Native relevance gate','',anchor_gate.to_markdown(index=False),'','## Local geometry and trajectory decisions','',evidence.to_markdown(index=False),'','## Interpretation requirements','',
'Inspect finite-difference convergence and numerical closure before interpreting Fisher residuals. A failed joint symmetry/optimizer identity test prevents a gauge interpretation even if an LR-only profile helps. A missing native gap makes the local mechanism an independent result, not an explanation of the reported advantage. Compare actual held-out retention loss alongside KL. Full fine-tuning and controlled pretraining remain necessary for population and long-horizon causal claims.','']
(OUT/'investigation_a_summary.md').write_text('\n'.join(summary_lines))

### Reading the final results
- Native gap and validated transplant + meaningful non-global residual + matched-movement utility advantage: proceed to full fine-tuning with the smallest adequate profile structure.
- Native gap but global/two-group fit is adequate and similarly useful: prioritize the simpler explanation; fine-grained geometry may not be necessary.
- Meaningful residual without utility advantage: the mechanism exists but is not an explanation of improved plasticity under these conditions.
- LR-only benefit without physical identity: report a norm-derived optimizer heuristic, not a causal gauge transplant.
- No native gap: revisit task/horizon/protocol, report local geometry separately. Historical pretraining effects, representation changes and attention structure are unresolved.

Record any departure from the declared settings as exploratory. The code does not fabricate results or claim exact replication of the original paper. No shape assertions are needed for these OLMo-2 weights; the metadata table serves provenance rather than generic defensive programming.